# GPR Field Data QC Viewer
**Lunar Leaper / pE PRO**

Load, visualize, and interactively process GPR data. Select a dataset and line using the dropdowns, then adjust processing parameters.

In [33]:
# -- Imports ------------------------------------------------------------------
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import gdp
from gdp.preprocessing.time_zero_correction import find_sample_above_threshold
from pathlib import Path

widgets.Widget.close_all()  # Clear any stale widget state

## 2. Select data
Use the dropdowns to choose a dataset folder and line. The data loads automatically on selection.

In [34]:
# Set the root directory for GPR data
GPR_ROOT = Path("../Data/GPR/Test data")

In [ ]:
# -- Dataset and line selection --------------------------------------------------
def get_datasets():
    # Relative path of each DT1 parent folder from GPR_ROOT
    return sorted(set(
        str(p.parent.relative_to(GPR_ROOT))
        for p in GPR_ROOT.rglob("*.DT1")
    ))

def get_lines(dataset):
    return sorted(p.name for p in (GPR_ROOT / dataset).glob("*.DT1"))

datasets = get_datasets()

w_dataset = widgets.Dropdown(
    options=datasets,
    value="Skull cave" if "Skull cave" in datasets else datasets[0],
    description="Dataset:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px")
)
w_line = widgets.Dropdown(
    options=get_lines(w_dataset.value),
    description="Line:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

def on_dataset_change(change):
    w_line.options = get_lines(change["new"])

w_dataset.observe(on_dataset_change, names="value")
display(widgets.HBox([w_dataset, w_line]))


In [36]:
# -- Processing --------------------------------------------------
out_load = widgets.Output()

# Set by load_data(), read by the processing cell
original_data = None
info = sfreq = nyquist = time_axis = dist_axis = None
freq = suggested_dewow = None

def load_data(_=None):
    global original_data, info, sfreq, nyquist, time_axis, dist_axis, freq, suggested_dewow
    data_file = GPR_ROOT / w_dataset.value / w_line.value
    out_load.clear_output(wait=True)
    with out_load:
        print(f"Loading {data_file}...")
        gpr_data = gdp.load_dt1(str(data_file))
        original_data = np.asarray(gpr_data.data, dtype=np.float64)
        info = gpr_data.info

        n_samples     = info['samples']
        time_window   = info['Total_time_window']  # ns
        sfreq         = n_samples / time_window * 1000  # MHz
        nyquist       = sfreq / 2
        ns_per_sample = time_window / n_samples
        time_axis     = np.linspace(0, time_window, n_samples)
        dist_axis     = np.linspace(info['Start_pos'], info['Final_pos'], original_data.shape[1])

        freq  = info['Freq']
        s_pos = info['Start_pos']
        f_pos = info['Final_pos']
        units = info['Pos_units']
        step  = info['Step_size']
        print(f"Loaded: {original_data.shape[0]} samples x {original_data.shape[1]} traces")
        print(f"  Antenna freq  : {freq} MHz")
        print(f"  Sampling rate : {sfreq:.0f} MHz  (Nyquist: {nyquist:.0f} MHz)")
        print(f"  Time window   : {time_window:.0f} ns  ({n_samples} samples, {ns_per_sample:.3f} ns/sample)")
        print(f"  Distance      : {s_pos:.1f} - {f_pos:.1f} {units}  (step {step} {units})")

        # Suggested dewow window: ~2 pulse periods (2 / antenna_freq in ns -> samples)
        suggested_dewow = round(2 / freq * 1000 / ns_per_sample)
        print(f"\n  Suggested dewow window : {suggested_dewow} samples  (~2 pulse periods at {freq} MHz)")

        tz_header    = info['TZ_at_pt']
        tz_header_ns = tz_header * ns_per_sample
        median_trace = np.median(original_data, axis=1)
        tz_gdp       = find_sample_above_threshold(median_trace, threshold=0.15)
        tz_gdp_ns    = tz_gdp * ns_per_sample
        diff_s       = tz_gdp - tz_header
        diff_ns      = tz_gdp_ns - tz_header_ns
        print(f"\n-- Time zero comparison --")
        print(f"  Header (TZ_at_pt)  : sample {tz_header:.2f}  ->  {tz_header_ns:.1f} ns")
        print(f"  gdp (median trace) : sample {tz_gdp}  ->  {tz_gdp_ns:.1f} ns")
        print(f"  Difference         : {diff_s:.2f} samples  ({diff_ns:.1f} ns)")

        # Update slider ranges -- defined in the processing cell
        if 'update_sliders' in globals():
            update_sliders(nyquist, tz_header, suggested_dewow)

w_line.observe(load_data, names="value")
w_dataset.observe(load_data, names="value")
load_data()
display(out_load)

Output()

## 3. Interactive processing

Adjust the sliders below to process your GPR data in real-time. Use the preview window to assess data quality before finalizing parameters.

In [37]:
# -- Processing function ------------------------------------------------------
def apply_processing(dewow_w, tzero_shift, flow, fhigh, gain_exp, stack_n):
    from gdp.preprocessing.filtering import dewow as dewow_fn, filter_data
    from gdp.preprocessing.gain import apply_gain as apply_gain_fn
    from scipy.ndimage import shift as ndshift

    processed = original_data.copy()

    # Stacking: average every stack_n consecutive traces before any filtering
    if stack_n > 1:
        n_samp, n_tr = processed.shape
        n_stacked = n_tr // stack_n
        processed = processed[:, :n_stacked * stack_n].reshape(n_samp, n_stacked, stack_n).mean(axis=2)

    processed = dewow_fn(processed, window_length=dewow_w)

    if tzero_shift != 0:
        processed = ndshift(processed, (tzero_shift, 0), order=1, mode='constant', cval=0)

    try:
        processed = filter_data(processed, (flow, fhigh), sfreq, 'bandpass', N=4)
    except Exception as e:
        print(f"Bandpass filter failed: {e}")

    try:
        processed, _ = apply_gain_fn(processed, sfreq, 'linear', exponent=gain_exp)
    except Exception as e:
        print(f"Gain failed: {e}")

    return processed

# -- Widget definitions -------------------------------------------------------
w_stack = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description="Stack (traces):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_dewow = widgets.IntSlider(
    value=suggested_dewow, min=1, max=150, step=1,
    description="Dewow window (samples):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_tzero = widgets.FloatSlider(
    value=-info['TZ_at_pt'], min=-200, max=100, step=0.01,
    description="Time zero shift (samples):",
    continuous_update=False,
    style={"description_width": "180px"},
    layout=widgets.Layout(width="480px")
)
w_flow = widgets.FloatSlider(
    value=0.2*freq, min=1, max=freq, step=1,
    description="Low cutoff (MHz):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_fhigh = widgets.FloatSlider(
    value=2*freq, min=freq, max=nyquist*0.95, step=1,
    description="High cutoff (MHz):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_gain = widgets.FloatSlider(
    value=1.0, min=0., max=3.5, step=0.1,
    description="Gain exponent:",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_xrange = widgets.FloatRangeSlider(
    value=[dist_axis[0], dist_axis[-1]],
    min=dist_axis[0], max=dist_axis[-1], step=0.1,
    description="X range (m):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".1f"
)
w_depth_toggle = widgets.ToggleButton(
    value=False,
    description="Show depth",
    button_style="",
    tooltip="Convert y-axis from two-way travel time (ns) to depth (m) using velocity below",
    layout=widgets.Layout(width="160px")
)
w_velocity = widgets.FloatSlider(
    value=0.13, min=0.05, max=0.3, step=0.005,
    description="Velocity (m/ns):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".3f"
)

out = widgets.Output()

# -- Slider update on dataset change ------------------------------------------
# ipywidgets validates min/max immediately. Reset min to 1 first so the new max
# can always be set safely regardless of which direction the antenna freq changes.
def update_sliders(nyquist, tz_header, suggested_dewow):
    _freq = info['Freq']
    w_flow.max   = _freq
    w_flow.value = 0.2 * _freq
    w_fhigh.min  = 1
    w_fhigh.max  = nyquist * 0.95
    w_fhigh.min  = _freq
    w_fhigh.value = 2 * _freq
    w_tzero.value = -tz_header
    w_dewow.value = suggested_dewow
    w_xrange.min  = dist_axis[0]
    w_xrange.max  = dist_axis[-1]
    w_xrange.value = [dist_axis[0], dist_axis[-1]]

# -- Plotting function --------------------------------------------------------
def plot_comparison(dewow_w, tzero_shift, flow, fhigh, gain_exp, show_depth, velocity, stack_n, xrange):
    from plotly.subplots import make_subplots

    processed = apply_processing(dewow_w, tzero_shift, flow, fhigh, gain_exp, stack_n)

    # Build dist_plot and orig_plot for stacked data
    if stack_n > 1:
        n_stacked = original_data.shape[1] // stack_n
        dist_plot = np.linspace(dist_axis[0], dist_axis[-1], n_stacked)
        n_samp, n_tr = original_data.shape
        orig_plot = original_data[:, :n_stacked * stack_n].reshape(n_samp, n_stacked, stack_n).mean(axis=2)
    else:
        dist_plot = dist_axis
        orig_plot = original_data

    # Crop to selected x range
    x0, x1 = xrange
    mask = (dist_plot >= x0) & (dist_plot <= x1)
    dist_plot  = dist_plot[mask]
    orig_plot  = orig_plot[:, mask]
    processed  = processed[:, mask]

    vmax_orig = np.percentile(np.abs(orig_plot), 99)
    vmax_proc = np.percentile(np.abs(processed), 99)

    if show_depth:
        y_axis  = time_axis * velocity / 2
        y_label = "Depth (m)"
        y_hover = "Depth: %{y:.2f} m"
    else:
        y_axis  = time_axis
        y_label = "Two-way travel time (ns)"
        y_hover = "Time: %{y:.1f} ns"

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"Original (stacked x{stack_n})", "Processed"),
        horizontal_spacing=0.14,
    )

    common = dict(x=dist_plot, y=y_axis, colorscale='Gray', showscale=True)

    fig.add_trace(go.Heatmap(
        **common,
        z=orig_plot,
        zmin=-vmax_orig, zmax=vmax_orig,
        colorbar=dict(x=0.42, thickness=15, len=0.9, title="Ampl."),
        hovertemplate=f"Dist: %{{x:.1f}} m<br>{y_hover}<br>Amplitude: %{{z:.2e}}<extra>Original</extra>"
    ), row=1, col=1)

    fig.add_trace(go.Heatmap(
        **common,
        z=processed,
        zmin=-vmax_proc, zmax=vmax_proc,
        colorbar=dict(x=1.00, thickness=15, len=0.9, title="Ampl."),
        hovertemplate=f"Dist: %{{x:.1f}} m<br>{y_hover}<br>Amplitude: %{{z:.2e}}<extra>Processed</extra>"
    ), row=1, col=2)

    yaxis_style = dict(title=y_label, showgrid=False, autorange="reversed")
    vel_str = f"  |  v = {velocity:.3f} m/ns" if show_depth else ""
    stack_str = f"  |  Stack: {stack_n}" if stack_n > 1 else ""

    fig.update_layout(
        title=dict(
            text=(
                f"GPR Radargram{stack_str}  |  Dewow: {dewow_w} samples  |  "
                f"TZero: {tzero_shift}  |  Filter: {flow}-{fhigh} MHz  |  Gain exp: {gain_exp}{vel_str}"
            ),
            font=dict(size=12)
        ),
        height=600,
        plot_bgcolor="white",
        paper_bgcolor="white",
        hovermode="closest",
        margin=dict(r=80),
        yaxis=yaxis_style,
        yaxis2={**yaxis_style, "matches": "y"},
    )
    fig.update_xaxes(title_text=f"Distance ({info['Pos_units']})", showgrid=False)
    fig.show()

    rms_orig = np.sqrt(np.mean(orig_plot**2))
    rms_proc = np.sqrt(np.mean(processed**2))
    warn = "  (gain > 1: inflated)" if gain_exp > 1 else ""
    print(f"\nRMS = sqrt(mean(amplitude^2)) -- signal energy check")
    print(f"  Original : {rms_orig:.3e}")
    print(f"  Processed: {rms_proc:.3e}{warn}")

# -- Wire up widgets ----------------------------------------------------------
def on_change(_):
    out.clear_output(wait=True)
    with out:
        plot_comparison(
            int(w_dewow.value), float(w_tzero.value),
            float(w_flow.value), float(w_fhigh.value), float(w_gain.value),
            bool(w_depth_toggle.value), float(w_velocity.value),
            int(w_stack.value), w_xrange.value
        )

for w in [w_stack, w_dewow, w_tzero, w_flow, w_fhigh, w_gain, w_xrange, w_depth_toggle, w_velocity]:
    w.observe(on_change, names="value")

# -- Display ------------------------------------------------------------------
display(widgets.VBox([
    widgets.HTML("<b>Stacking</b>"),
    w_stack,
    widgets.HTML("<b>Processing parameters</b>"),
    w_dewow, w_tzero, w_flow, w_fhigh, w_gain,
    widgets.HTML("<b>View</b>"),
    w_xrange,
    widgets.HBox([w_depth_toggle, w_velocity]),
]))
display(out)
on_change(None)

Output()